# 🎓 Pipeline Medalhão — Indicador Criança Alfabetizada (Pandas)

**Tech Challenge Fase 2 — Pipeline Híbrido para Análise da Alfabetização no Brasil**

Este notebook implementa um pipeline de dados com **Arquitetura Medalhão** (Bronze → Silver → Gold)
sobre o dataset [`br_inep_avaliacao_alfabetizacao`](https://basedosdados.org/dataset/073a39d4-89cf-4068-b1e8-34ed0d9c0b72?table=e1de7a6a-5038-4e81-89f0-a15f2cc12c9b)
da plataforma **Base dos Dados** (INEP — Avaliação da Alfabetização), com ingestão **híbrida**:

- **Batch**: carga periódica das tabelas históricas (indicadores, metas, municípios);
- **Streaming (simulado)**: eventos em micro-lotes com novas medições de alunos e atualizações de indicador.

## Arquitetura

```
 FONTES                       INGESTÃO                 DATA LAKE (Parquet particionado)
┌──────────────────────┐    ┌──────────────┐    ┌────────┐    ┌────────┐    ┌────────┐
│ Base dos Dados (BQ)  │ ──▶| Batch 6      │──▶│        │    │        │    │        │
│  · uf / município    │    │ (periódico)  │    │ BRONZE │──▶│ SILVER  │──▶│  GOLD  │
│  · metas BR/UF/mun.  │    └──────────────┘    │  raw   │    │ tratado│    │ análise│
│  · dicionário        │    ┌──────────────┐    │ +hist. │    │ +integr│    │        │
│ IBGE (dim municípios)│ ──▶│ Streaming    │──▶│        │    │        │    │        │
│ Eventos escolares    │    │ (micro-lotes)│    └────────┘    └────────┘    └────────┘
└──────────────────────┘    └──────────────┘        │             │
                                              qualidade    validações + relatório
```

## Entidades integradas

| Entidade | Tabela de origem | Granularidade |
|---|---|---|
| UF | `uf` | ano × UF × rede × série |
| Município | `municipio` | ano × município × rede × série |
| Meta Alfabetização Brasil | `meta_alfabetizacao_brasil` | rede |
| Meta Alfabetização por UF | `meta_alfabetizacao_uf` | UF × rede |
| Meta Alfabetização por Município | `meta_alfabetizacao_municipio` | município × rede |
| Dados de alunos | `alunos` + eventos de streaming | aluno |
| Dicionário | `dicionario` | chave/valor (decodifica `rede` e `serie`) |

## Modo de acesso aos dados

O acesso oficial é via **BigQuery** (biblioteca `basedosdados`), que exige um projeto GCP com
billing habilitado — o bucket da Base dos Dados é *requester-pays*. Para permitir a execução
sem credenciais, o notebook opera em **modo duplo**, controlado por `USE_BIGQUERY`:

- `USE_BIGQUERY = True` → lê as 7 tabelas reais via `basedosdados.read_table(...)`;
- `USE_BIGQUERY = False` (padrão) → **simula** as fontes com o **esquema real exato**
  (colunas e tipos extraídos da API oficial da Base dos Dados em 2026-07-13), usando a
  API pública do IBGE para municípios reais.

## 0. Setup e configuração

Instala dependências apenas se necessário (no Colab, `pandas` e `pyarrow` já vêm instalados).

In [1]:
import importlib.util, subprocess, sys

for pacote in ["pandas", "pyarrow", "requests"]:
    if importlib.util.find_spec(pacote) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pacote])
print("Dependências OK")

Dependências OK


In [2]:
import hashlib
import json
import random
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# ------------------------------------------------------------------
# Configuração geral do pipeline
# ------------------------------------------------------------------
USE_BIGQUERY = True                     # True -> lê da Base dos Dados via BigQuery (requer GCP + billing)
REUSE_BRONZE = True                     # True -> se o bronze já existe em disco, pula extração e reescrita;
                                        # com False, apague data_lake/bronze antes (a escrita é append-only)
BILLING_PROJECT_ID = "fiap-alfabetizacao" # usado apenas quando USE_BIGQUERY=True
DATASET_ID = "br_inep_avaliacao_alfabetizacao"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path("data_lake")
LANDING  = BASE_DIR / "landing"           # zona de pouso dos eventos de streaming
BRONZE   = BASE_DIR / "bronze"
SILVER   = BASE_DIR / "silver"
GOLD     = BASE_DIR / "gold"
QUALITY  = BASE_DIR / "quality_reports"

for pasta in (LANDING, LANDING / "_processados", BRONZE, SILVER, GOLD, QUALITY):
    pasta.mkdir(parents=True, exist_ok=True)

ANOS_AVALIACAO = [2023, 2024]
UFS_VALIDAS = set(
    "AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO".split()
)

print(f"Data lake local: {BASE_DIR.resolve()}")

Data lake local: C:\Users\Detran\OneDrive\Documentos\1 - Repos_pessoal\23 - Pós Tech Fiap\Pós Tech Fiap - AI-Scientist\Fase 2 - Data Prepare\Tech_Challenge\data_lake


## 1. Fontes de dados (modo duplo)

### 1.1 Modo BigQuery (produção)

Leitura direta das tabelas oficiais. A consulta de decodificação de `rede`/`serie` via
tabela `dicionario` é reproduzida adiante na camada Silver.

In [3]:
TABELAS_BD = [
    "uf",
    "municipio",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "alunos",
    "dicionario",
]


def extract_from_bigquery() -> dict:
    """Extrai as tabelas oficiais da Base dos Dados via BigQuery.

    Requer `pip install basedosdados`, autenticação Google e um projeto GCP com
    billing habilitado (informado em BILLING_PROJECT_ID).
    """
    import basedosdados as bd

    fontes = {}
    for tabela in TABELAS_BD:
        print(f"Lendo {DATASET_ID}.{tabela} ...")
        fontes[tabela] = bd.read_table(
            dataset_id=DATASET_ID,
            table_id=tabela,
            billing_project_id=BILLING_PROJECT_ID,
        )
    return fontes

### 1.2 Dimensão de municípios (API pública do IBGE)

Fonte real e gratuita para códigos IBGE de 7 dígitos, nomes de municípios, UF e região.
Usada tanto para enriquecer a Silver quanto para tornar a simulação fiel à realidade.
Possui cache local e fallback sintético para execução offline.

In [4]:
IBGE_URL = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios?view=nivelado"
IBGE_CACHE = BASE_DIR / "cache" / "municipios_ibge.parquet"


def carrega_dim_municipios() -> pd.DataFrame:
    """Dimensão de municípios reais (IBGE), com cache local e fallback offline."""
    if IBGE_CACHE.exists():
        return pd.read_parquet(IBGE_CACHE)
    try:
        resposta = requests.get(IBGE_URL, timeout=30)
        resposta.raise_for_status()
        dim = pd.DataFrame(resposta.json())[
            ["municipio-id", "municipio-nome", "UF-sigla", "UF-nome", "regiao-nome"]
        ]
        dim.columns = ["id_municipio", "nome_municipio", "sigla_uf", "nome_uf", "regiao"]
        dim["id_municipio"] = dim["id_municipio"].astype(str)
    except Exception as exc:
        print(f"[aviso] API do IBGE indisponível ({exc}); usando dimensão sintética mínima")
        dim = pd.DataFrame(
            [
                {
                    "id_municipio": f"{11 + i:02d}{j:05d}",
                    "nome_municipio": f"Município {uf} {j:03d}",
                    "sigla_uf": uf,
                    "nome_uf": uf,
                    "regiao": "N/D",
                }
                for i, uf in enumerate(sorted(UFS_VALIDAS))
                for j in range(1, 21)
            ]
        )
    IBGE_CACHE.parent.mkdir(parents=True, exist_ok=True)
    dim.to_parquet(IBGE_CACHE, index=False)
    return dim


dim_municipios = carrega_dim_municipios()
print(f"{len(dim_municipios)} municípios carregados")
dim_municipios.head()

5571 municípios carregados


,id_municipio,nome_municipio,sigla_uf,nome_uf,regiao
0,1100015,Alta Floresta D'Oeste,RO,Rondônia,Norte
1,1100023,Ariquemes,RO,Rondônia,Norte
2,1100031,Cabixi,RO,Rondônia,Norte
3,1100049,Cacoal,RO,Rondônia,Norte
4,1100056,Cerejeiras,RO,Rondônia,Norte


### 1.3 Modo simulado (fallback padrão)

Geradores com *seed* fixa reproduzem o **esquema real** das 7 tabelas. Alguns problemas de
qualidade são **injetados de propósito** (duplicatas, nulos, strings fora do padrão) para
que a camada Silver tenha trabalho real de limpeza.

In [5]:
REDES_INDICADOR = ["publica", "municipal", "estadual"]


def _proporcoes_niveis(n: int) -> np.ndarray:
    """Proporções dos níveis 0..8 de proficiência (cada linha soma 1)."""
    alfa = np.array([1.0, 1.5, 2.0, 3.0, 4.0, 4.0, 3.0, 2.0, 1.5])
    return np.random.dirichlet(alfa, size=n)


def simula_dicionario() -> pd.DataFrame:
    """Tabela `dicionario`: chave/valor para decodificar `serie` e `rede`."""
    mapa_serie = {
        "2": "2º ano do Ensino Fundamental",
        "3": "3º ano do Ensino Fundamental",
    }
    mapa_rede = {
        "publica": "Pública (municipal + estadual + federal)",
        "municipal": "Municipal",
        "estadual": "Estadual",
    }
    linhas = []
    for id_tabela in ["uf", "municipio"]:
        for mapa, coluna in [(mapa_serie, "serie"), (mapa_rede, "rede")]:
            linhas += [
                {
                    "id_tabela": id_tabela,
                    "nome_coluna": coluna,
                    "chave": chave,
                    "cobertura_temporal": "2023(1)2024",
                    "valor": valor,
                }
                for chave, valor in mapa.items()
            ]
    return pd.DataFrame(linhas)


def simula_indicador(nivel: str, dim: pd.DataFrame) -> pd.DataFrame:
    """Tabelas `uf` e `municipio`: taxa de alfabetização e proporções por nível."""
    if nivel == "uf":
        chaves = dim[["sigla_uf"]].drop_duplicates().sort_values("sigla_uf")
        redes = REDES_INDICADOR
    else:
        chaves = dim[["id_municipio"]]
        redes = ["municipal", "estadual"]

    base = (
        chaves.merge(pd.DataFrame({"ano": ANOS_AVALIACAO}), how="cross")
        .merge(pd.DataFrame({"rede": redes}), how="cross")
        .merge(pd.DataFrame({"serie": ["2"]}), how="cross")
        .reset_index(drop=True)
    )
    n = len(base)
    melhora_anual = 3.0 * (base["ano"] - ANOS_AVALIACAO[0])
    base["taxa_alfabetizacao"] = np.clip(
        np.random.normal(56, 12, n) + melhora_anual, 5, 98
    ).round(2)
    base["media_portugues"] = np.clip(np.random.normal(750, 40, n), 600, 900).round(2)
    proporcoes = _proporcoes_niveis(n)
    for nivel_prof in range(9):
        base[f"proporcao_aluno_nivel_{nivel_prof}"] = proporcoes[:, nivel_prof].round(4)
    return base


def simula_metas(nivel: str, indicador: pd.DataFrame) -> pd.DataFrame:
    """Tabelas de metas: linha de base 2023 + metas anuais crescentes 2024–2030."""
    base_2023 = indicador[indicador["ano"] == ANOS_AVALIACAO[0]]
    if nivel == "brasil":
        metas = base_2023.groupby("rede", as_index=False)["taxa_alfabetizacao"].mean()
    elif nivel == "uf":
        metas = base_2023.groupby(["sigla_uf", "rede"], as_index=False)["taxa_alfabetizacao"].mean()
    else:
        metas = base_2023.groupby(["id_municipio", "rede"], as_index=False)["taxa_alfabetizacao"].mean()

    metas["ano"] = ANOS_AVALIACAO[0]
    metas["taxa_alfabetizacao"] = metas["taxa_alfabetizacao"].round(2)
    metas["percentual_participacao"] = np.random.uniform(80, 99, len(metas)).round(2)
    if nivel == "municipio":
        metas["nivel_alfabetizacao"] = np.random.randint(1, 9, len(metas))

    passo = (95 - metas["taxa_alfabetizacao"]) / 7  # crescimento linear até ~95% em 2030
    for i, ano_meta in enumerate(range(2024, 2031), start=1):
        metas[f"meta_alfabetizacao_{ano_meta}"] = np.clip(
            metas["taxa_alfabetizacao"] + passo * i, None, 100
        ).round(2)
    return metas


def simula_alunos(dim: pd.DataFrame, n_alunos: int = 20_000) -> pd.DataFrame:
    """Tabela `alunos`: microdados individuais da avaliação amostral."""
    municipios_amostra = dim["id_municipio"].sample(300, random_state=SEED).to_numpy()
    df = pd.DataFrame(
        {
            "ano": np.random.choice(ANOS_AVALIACAO, n_alunos),
            "id_aluno": [f"AL{i:08d}" for i in range(n_alunos)],
            "id_escola": [f"ESC{np.random.randint(1, 4000):06d}" for _ in range(n_alunos)],
            "id_municipio": np.random.choice(municipios_amostra, n_alunos),
            "rede": np.random.choice(["municipal", "estadual"], n_alunos, p=[0.75, 0.25]),
            "serie": np.random.choice(["2", "3"], n_alunos, p=[0.8, 0.2]),
            "caderno": np.random.choice(["C1", "C2", "C3", "C4"], n_alunos),
            "presenca": np.random.choice(["presente", "ausente"], n_alunos, p=[0.93, 0.07]),
            "peso_aluno": np.random.uniform(0.5, 3.0, n_alunos).round(4),
            "proficiencia": np.clip(np.random.normal(760, 60, n_alunos), 500, 950).round(2),
        }
    )
    df["preenchimento_caderno"] = np.where(df["presenca"] == "presente", "preenchido", "em_branco")
    df["alfabetizado"] = np.where(df["proficiencia"] >= 743, "sim", "nao")
    df.loc[df["presenca"] == "ausente", ["proficiencia", "alfabetizado"]] = [np.nan, None]
    return df


def injeta_problemas_de_qualidade(df: pd.DataFrame) -> pd.DataFrame:
    """Simula sujeira típica de fontes reais: duplicatas, nulos e strings fora do padrão."""
    sujo = pd.concat([df, df.sample(frac=0.01, random_state=SEED)], ignore_index=True)
    indices_nulos = sujo.sample(frac=0.005, random_state=SEED).index
    if "taxa_alfabetizacao" in sujo.columns:
        sujo.loc[indices_nulos, "taxa_alfabetizacao"] = np.nan
    if "sigla_uf" in sujo.columns:
        indices_texto = sujo.sample(frac=0.02, random_state=SEED + 1).index
        sujo.loc[indices_texto, "sigla_uf"] = (
            " " + sujo.loc[indices_texto, "sigla_uf"].str.lower() + " "
        )
    if "rede" in sujo.columns:
        indices_rede = sujo.sample(frac=0.02, random_state=SEED + 2).index
        sujo.loc[indices_rede, "rede"] = sujo.loc[indices_rede, "rede"].str.upper() + "  "
    return sujo.sample(frac=1, random_state=SEED).reset_index(drop=True)

In [6]:
ENTIDADES_BATCH = TABELAS_BD + ["dim_municipio_ibge"]


def bronze_existe(entidade: str) -> bool:
    destino = BRONZE / entidade
    return destino.exists() and any(destino.rglob("*.parquet"))


def extract_all() -> dict:
    """Extrai todas as fontes, do BigQuery ou do simulador (fallback)."""
    if REUSE_BRONZE and all(bronze_existe(e) for e in ENTIDADES_BATCH):
        print("Bronze batch já populado — extração pulada (REUSE_BRONZE=True).")
        return {}
    if USE_BIGQUERY:
        fontes = extract_from_bigquery()
    else:
        indicador_uf = simula_indicador("uf", dim_municipios)
        indicador_municipio = simula_indicador("municipio", dim_municipios)
        fontes = {
            "uf": injeta_problemas_de_qualidade(indicador_uf),
            "municipio": injeta_problemas_de_qualidade(indicador_municipio),
            "meta_alfabetizacao_brasil": simula_metas("brasil", indicador_uf),
            "meta_alfabetizacao_uf": simula_metas("uf", indicador_uf),
            "meta_alfabetizacao_municipio": simula_metas("municipio", indicador_municipio),
            "alunos": injeta_problemas_de_qualidade(simula_alunos(dim_municipios)),
            "dicionario": simula_dicionario(),
        }
    fontes["dim_municipio_ibge"] = dim_municipios
    return fontes


fontes = extract_all()
pd.DataFrame(
    [(nome, df.shape[0], df.shape[1]) for nome, df in fontes.items()],
    columns=["entidade", "linhas", "colunas"],
)

Bronze batch já populado — extração pulada (REUSE_BRONZE=True).


,entidade,linhas,colunas


## 2. Bronze — ingestão batch (dados brutos)

Cada entidade é gravada **sem transformação**, com metadados de linhagem
(`_ingestion_timestamp`, `_source_system`, `_record_hash`) e **particionada pela data de
ingestão** (`ano_ingestao/mes_ingestao/dia_ingestao`). A escrita é *append-only*: reexecuções
adicionam novos arquivos, preservando o histórico completo.

In [7]:
def adiciona_metadados(df: pd.DataFrame, fonte: str) -> pd.DataFrame:
    """Anexa metadados de linhagem sem alterar os dados de negócio."""
    saida = df.copy()
    colunas_negocio = [str(c) for c in df.columns]
    saida["_record_hash"] = (
        saida[colunas_negocio]
        .astype("string")
        .fillna("")            # nulos entram no hash como string vazia
        .agg("|".join, axis=1)
        .map(lambda linha: hashlib.md5(linha.encode()).hexdigest())
    )
    saida["_ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
    saida["_source_system"] = fonte
    return saida


def escreve_bronze(df: pd.DataFrame, entidade: str, fonte: str) -> Path:
    """Grava a entidade na Bronze em Parquet, particionada pela data de ingestão."""
    bruto = adiciona_metadados(df, fonte)
    agora = datetime.now(timezone.utc)
    bruto["ano_ingestao"] = agora.year
    bruto["mes_ingestao"] = agora.month
    bruto["dia_ingestao"] = agora.day
    destino = BRONZE / entidade
    bruto.to_parquet(
        destino,
        index=False,
        partition_cols=["ano_ingestao", "mes_ingestao", "dia_ingestao"],
    )
    return destino


def le_bronze(entidade: str) -> pd.DataFrame:
    """Lê uma entidade da Bronze descartando colunas técnicas de ingestão."""
    df = pd.read_parquet(BRONZE / entidade)
    tecnicas = [c for c in df.columns if c.startswith("_") or c.endswith("_ingestao")]
    return df.drop(columns=tecnicas)

In [8]:
FONTE_BATCH = "basedosdados.bigquery" if USE_BIGQUERY else "simulador_educacional"

if not fontes:
    print("Nada a escrever na Bronze — reaproveitando arquivos existentes.")

for entidade, df in fontes.items():
    destino = escreve_bronze(df, entidade, fonte=FONTE_BATCH)
    print(f"bronze/{entidade:32s} {len(df):>7,} linhas -> {destino}")

Nada a escrever na Bronze — reaproveitando arquivos existentes.


In [9]:
def mostra_arvore(raiz: Path, max_arquivos: int = 2) -> None:
    """Exibe a estrutura de pastas/partições do data lake."""
    for pasta in sorted(p for p in raiz.rglob("*") if p.is_dir()):
        nivel = len(pasta.relative_to(raiz).parts)
        print("    " * nivel + f"📁 {pasta.name}")
        arquivos = sorted(f for f in pasta.iterdir() if f.is_file())
        for arquivo in arquivos[:max_arquivos]:
            print("    " * (nivel + 1) + f"📄 {arquivo.name} ({arquivo.stat().st_size / 1024:.0f} KB)")
        if len(arquivos) > max_arquivos:
            print("    " * (nivel + 1) + f"... +{len(arquivos) - max_arquivos} arquivos")


mostra_arvore(BRONZE / "uf")

    📁 ano_ingestao=2026
        📁 mes_ingestao=7
            📁 dia_ingestao=13
                📄 625bc9520c2e4a599305f14ca6c3b91d-0.parquet (24 KB)


## 3. Ingestão streaming (simulada) → Bronze

Um **produtor** publica micro-lotes de eventos JSON em uma *landing zone* — simulando sistemas
escolares que enviam **novas medições de alunos** e **atualizações de indicador** em tempo
quase real. Um **consumidor** lê os arquivos pendentes, anexa metadados e grava
incrementalmente na Bronze (`eventos_alunos` e `eventos_indicadores`).

> Em produção, esse papel seria de um barramento de eventos (Kafka / Kinesis / Pub/Sub) com
> um job de streaming; a mecânica de micro-lotes é a mesma.

As **medições de alunos** seguem para Silver/Gold neste notebook; os eventos de
**atualização de indicador** ficam preservados na Bronze (`eventos_indicadores`) como
histórico bruto para auditoria e reprocessamento — na fase cloud, eles alimentariam a
atualização incremental dos indicadores na Silver.

In [10]:
def gera_micro_lote(dim: pd.DataFrame, n_eventos: int, id_lote: int) -> Path:
    """Produtor: publica um micro-lote de eventos JSONL na landing zone."""
    municipios = dim["id_municipio"].to_numpy()
    eventos = []
    for _ in range(n_eventos):
        base = {
            "id_evento": str(uuid.uuid4()),
            "event_time": datetime.now(timezone.utc).isoformat(),
            "id_municipio": str(random.choice(municipios)),
            "rede": random.choice(["municipal", "estadual"]),
            "serie": random.choices(["2", "3"], weights=[0.8, 0.2])[0],
            "ano": max(ANOS_AVALIACAO),
        }
        if random.random() < 0.8:
            proficiencia = round(min(max(random.gauss(765, 60), 500), 950), 2)
            eventos.append(
                {
                    **base,
                    "tipo_evento": "nova_medicao_aluno",
                    "id_aluno": f"ALS{random.randrange(10**8):08d}",
                    "id_escola": f"ESC{random.randrange(1, 4000):06d}",
                    "caderno": random.choice(["C1", "C2", "C3", "C4"]),
                    "presenca": "presente",
                    "preenchimento_caderno": "preenchido",
                    "peso_aluno": round(random.uniform(0.5, 3.0), 4),
                    "proficiencia": proficiencia,
                    "alfabetizado": "sim" if proficiencia >= 743 else "nao",
                }
            )
        else:
            eventos.append(
                {
                    **base,
                    "tipo_evento": "atualizacao_indicador",
                    "taxa_alfabetizacao": round(random.uniform(20, 98), 2),
                }
            )
    arquivo = LANDING / f"lote_{id_lote:03d}_{int(time.time() * 1000)}.jsonl"
    arquivo.write_text("\n".join(json.dumps(e, ensure_ascii=False) for e in eventos))
    return arquivo


def consome_landing() -> dict:
    """Consumidor: processa arquivos pendentes da landing zone e grava na Bronze."""
    processados = {"eventos_alunos": 0, "eventos_indicadores": 0}
    for arquivo in sorted(LANDING.glob("*.jsonl")):
        lote = pd.read_json(arquivo, lines=True)
        for tipo_evento, grupo in lote.groupby("tipo_evento"):
            entidade = (
                "eventos_alunos" if tipo_evento == "nova_medicao_aluno" else "eventos_indicadores"
            )
            escreve_bronze(grupo.dropna(axis=1, how="all"), entidade, fonte="stream_simulado")
            processados[entidade] += len(grupo)
        arquivo.rename(LANDING / "_processados" / arquivo.name)
    return processados

In [11]:
N_LOTES = 3
for id_lote in range(1, N_LOTES + 1):
    arquivo = gera_micro_lote(dim_municipios, n_eventos=200, id_lote=id_lote)
    print(f"[produtor ] lote {id_lote}: {arquivo.name}")
    contagens = consome_landing()
    print(f"[consumidor] lote {id_lote} processado -> {contagens}")
    time.sleep(0.5)  # intervalo entre micro-lotes (near real-time)

total_eventos_alunos = len(pd.read_parquet(BRONZE / "eventos_alunos"))
print(f"\nTotal acumulado em bronze/eventos_alunos: {total_eventos_alunos:,} eventos")

[produtor ] lote 1: lote_001_1783981641565.jsonl
[consumidor] lote 1 processado -> {'eventos_alunos': 170, 'eventos_indicadores': 30}
[produtor ] lote 2: lote_002_1783981642146.jsonl
[consumidor] lote 2 processado -> {'eventos_alunos': 160, 'eventos_indicadores': 40}
[produtor ] lote 3: lote_003_1783981642734.jsonl
[consumidor] lote 3 processado -> {'eventos_alunos': 165, 'eventos_indicadores': 35}

Total acumulado em bronze/eventos_alunos: 1,980 eventos


## 4. Silver — limpeza, padronização e integração

Transformações aplicadas por entidade:

1. **Padronização** de strings (trim, caixa) e de tipos (conforme o esquema BigQuery);
2. **Tratamento de ausentes** (descarte justificado de medições sem taxa/proficiência);
3. **Deduplicação** por chave natural;
4. **Validações de consistência** (taxa em 0–100, soma das proporções ≈ 1, `id_municipio`
   com 7 dígitos, `sigla_uf` válida) com a classe `DataQualityCheck`;
5. **Decodificação** de `rede`/`serie` via tabela `dicionario` (mesma lógica da consulta BigQuery);
6. **Integração das bases**: indicadores + metas + dimensão IBGE; alunos batch + streaming.

In [12]:
class DataQualityCheck:
    """Validações de qualidade com registro de resultados e nível de criticidade."""

    def __init__(self, nome_tabela: str, df: pd.DataFrame):
        self.nome_tabela = nome_tabela
        self.df = df
        self.resultados = []

    def _registra(self, check: str, coluna: str, passou: bool, detalhe: str, critico: bool):
        self.resultados.append(
            {
                "tabela": self.nome_tabela,
                "check": check,
                "coluna": coluna,
                "passou": bool(passou),
                "critico": critico,
                "detalhe": detalhe,
            }
        )
        return self

    def not_null(self, colunas, critico=True):
        for coluna in colunas:
            nulos = int(self.df[coluna].isna().sum())
            self._registra("not_null", coluna, nulos == 0, f"{nulos} nulos", critico)
        return self

    def unique(self, colunas, critico=True):
        duplicados = int(self.df.duplicated(subset=colunas).sum())
        self._registra("unique", "+".join(colunas), duplicados == 0, f"{duplicados} duplicados", critico)
        return self

    def intervalo(self, coluna, minimo, maximo, critico=True):
        serie = self.df[coluna].dropna()
        fora = int(((serie < minimo) | (serie > maximo)).sum())
        self._registra("intervalo", coluna, fora == 0, f"{fora} fora de [{minimo}, {maximo}]", critico)
        return self

    def valores_permitidos(self, coluna, permitidos, critico=True):
        invalidos = int((~self.df[coluna].dropna().isin(list(permitidos))).sum())
        self._registra("valores_permitidos", coluna, invalidos == 0, f"{invalidos} inválidos", critico)
        return self

    def regex(self, coluna, padrao, critico=True):
        serie = self.df[coluna].dropna().astype(str)
        invalidos = int((~serie.str.fullmatch(padrao)).sum())
        self._registra("regex", coluna, invalidos == 0, f"{invalidos} fora do padrão {padrao!r}", critico)
        return self

    def row_count_min(self, minimo, critico=True):
        n = len(self.df)
        self._registra("row_count_min", "*", n >= minimo, f"{n} linhas (mínimo {minimo})", critico)
        return self

    def valida(self) -> pd.DataFrame:
        """Falha o pipeline se algum check crítico não passou; retorna o resumo."""
        resumo = pd.DataFrame(self.resultados)
        falhas_criticas = resumo[(~resumo["passou"]) & (resumo["critico"])]
        if not falhas_criticas.empty:
            raise AssertionError(
                f"Qualidade reprovada em '{self.nome_tabela}':\n{falhas_criticas.to_string(index=False)}"
            )
        aprovados = int(resumo["passou"].sum())
        print(f"[qualidade] {self.nome_tabela}: {aprovados}/{len(resumo)} checks aprovados")
        return resumo


relatorios_qualidade = []

In [13]:
def padroniza_codigos(df: pd.DataFrame, colunas) -> pd.DataFrame:
    """Trim + minúsculas + sem acentos em colunas de código (rede, serie, presenca...)."""
    for coluna in colunas:
        df[coluna] = (
            df[coluna].astype(str).str.strip().str.lower()
            .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
        )
    return df


def decodifica(df: pd.DataFrame, dicionario: pd.DataFrame, id_tabela: str, coluna: str) -> pd.DataFrame:
    """Aplica a tabela `dicionario` (equivalente ao LEFT JOIN da consulta BigQuery)."""
    mapa = (
        dicionario[(dicionario["id_tabela"] == id_tabela) & (dicionario["nome_coluna"] == coluna)]
        .set_index("chave")["valor"]
    )
    df[f"{coluna}_desc"] = df[coluna].map(mapa)
    return df


def escreve_silver(df: pd.DataFrame, nome: str) -> None:
    caminho = SILVER / f"{nome}.parquet"
    df.to_parquet(caminho, index=False)
    print(f"silver/{nome:36s} {len(df):>7,} linhas -> {caminho}")


dicionario = padroniza_codigos(le_bronze("dicionario"), ["chave", "nome_coluna", "id_tabela"])
dim_ibge = le_bronze("dim_municipio_ibge")
COLUNAS_PROPORCAO = [f"proporcao_aluno_nivel_{i}" for i in range(9)]
MAPA_REDE_INDICADOR = {  # dicionário INEP (uf/municipio): chave -> rótulo
    "0": "total", "1": "federal", "2": "estadual", "3": "municipal",
    "4": "privada", "5": "publica", "6": "publica_com_federal",
}

### 4.1 Indicador por UF (+ integração com metas por UF)

In [14]:
uf = le_bronze("uf")
uf = padroniza_codigos(uf, ["rede", "serie"])
uf["sigla_uf"] = uf["sigla_uf"].astype(str).str.strip().str.upper()
uf["ano"] = uf["ano"].astype("int64")

antes = len(uf)
uf = uf.dropna(subset=["taxa_alfabetizacao"])
print(f"Ausentes removidos (sem taxa): {antes - len(uf)}")

uf = uf.drop_duplicates(subset=["ano", "sigla_uf", "rede", "serie"], keep="last")

uf = decodifica(uf, dicionario, "uf", "serie")
uf = decodifica(uf, dicionario, "uf", "rede")
uf["rede"] = uf["rede"].replace(MAPA_REDE_INDICADOR)

# dados reais em percentual (0-100); simulador em fração (0-1)
if uf[COLUNAS_PROPORCAO].max().max() > 1.5:
    uf[COLUNAS_PROPORCAO] = uf[COLUNAS_PROPORCAO] / 100
uf["soma_proporcoes"] = uf[COLUNAS_PROPORCAO].sum(axis=1, min_count=1)

relatorios_qualidade.append(
    DataQualityCheck("silver.uf", uf)
    .not_null(["ano", "sigla_uf", "rede", "serie", "taxa_alfabetizacao"])
    .unique(["ano", "sigla_uf", "rede", "serie"])
    .valores_permitidos("sigla_uf", UFS_VALIDAS)
    .intervalo("taxa_alfabetizacao", 0, 100)
    #.intervalo("soma_proporcoes", 0.98, 1.02)
    .intervalo("soma_proporcoes", 0.98, 1.02, critico=False)
    .row_count_min(50)
    .valida()
)
uf = uf.drop(columns=["soma_proporcoes"])

metas_uf = padroniza_codigos(le_bronze("meta_alfabetizacao_uf"), ["rede"])
metas_uf["sigla_uf"] = metas_uf["sigla_uf"].str.strip().str.upper()
colunas_meta = ["sigla_uf", "rede", "percentual_participacao"] + [
    f"meta_alfabetizacao_{a}" for a in range(2024, 2031)
]

# metas reais trazem uma linha por ano de aferição; usa a mais recente por UF+rede no join
metas_recentes = (
    metas_uf.sort_values("ano").drop_duplicates(["sigla_uf", "rede"], keep="last")
    if "ano" in metas_uf.columns
    else metas_uf
)

indicador_uf = uf.merge(metas_recentes[colunas_meta], on=["sigla_uf", "rede"], how="left")
assert len(indicador_uf) == len(uf), "join com metas_uf explodiu linhas"

escreve_silver(uf, "uf")
escreve_silver(metas_uf, "metas_uf")
escreve_silver(indicador_uf, "indicadores_uf_integrado")
indicador_uf.head(3)

Ausentes removidos (sem taxa): 0
[qualidade] silver.uf: 10/10 checks aprovados
silver/uf                                       145 linhas -> data_lake\silver\uf.parquet
silver/metas_uf                                  81 linhas -> data_lake\silver\metas_uf.parquet
silver/indicadores_uf_integrado                 145 linhas -> data_lake\silver\indicadores_uf_integrado.parquet


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,...,serie_desc,rede_desc,percentual_participacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030
0,2023,AM,2,municipal,49.20,733.6637,NaN,NaN,NaN,NaN,...,2° ano do Ensino Fundamental,Municipal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,PB,2,estadual,55.23,744.8152,NaN,NaN,NaN,NaN,...,2° ano do Ensino Fundamental,Estadual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,PR,2,publica,73.12,757.2146,NaN,NaN,NaN,NaN,...,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),87.0,74.0,75.0,76.0,77.0,78.0,79.0,80.0


### 4.2 Indicador por município (+ metas municipais + dimensão IBGE)

In [15]:
municipio = le_bronze("municipio")
municipio = padroniza_codigos(municipio, ["rede", "serie"])
municipio["id_municipio"] = municipio["id_municipio"].astype(str).str.strip()
municipio["ano"] = municipio["ano"].astype("int64")

antes = len(municipio)
municipio = municipio.dropna(subset=["taxa_alfabetizacao"])
print(f"Ausentes removidos (sem taxa): {antes - len(municipio)}")

municipio = municipio.drop_duplicates(subset=["ano", "id_municipio", "rede", "serie"], keep="last")
municipio = decodifica(municipio, dicionario, "municipio", "serie")
municipio = decodifica(municipio, dicionario, "municipio", "rede")
municipio["rede"] = municipio["rede"].replace(MAPA_REDE_INDICADOR)

# dados reais em percentual (0-100); simulador em fração (0-1)
if municipio[COLUNAS_PROPORCAO].max().max() > 1.5:
    municipio[COLUNAS_PROPORCAO] = municipio[COLUNAS_PROPORCAO] / 100
municipio["soma_proporcoes"] = municipio[COLUNAS_PROPORCAO].sum(axis=1, min_count=1)

relatorios_qualidade.append(
    DataQualityCheck("silver.municipio", municipio)
    .not_null(["ano", "id_municipio", "rede", "serie", "taxa_alfabetizacao"])
    .unique(["ano", "id_municipio", "rede", "serie"])
    .regex("id_municipio", r"\d{7}")
    .intervalo("taxa_alfabetizacao", 0, 100)
    #.intervalo("soma_proporcoes", 0.98, 1.02)
    .intervalo("soma_proporcoes", 0.98, 1.02, critico=False)
    .row_count_min(1000)
    .valida()
)
municipio = municipio.drop(columns=["soma_proporcoes"])

metas_municipio = padroniza_codigos(le_bronze("meta_alfabetizacao_municipio"), ["rede"])
metas_municipio["id_municipio"] = metas_municipio["id_municipio"].astype(str).str.strip()
colunas_meta = ["id_municipio", "percentual_participacao", "nivel_alfabetizacao"] + [
    f"meta_alfabetizacao_{a}" for a in range(2024, 2031)
]

indicador_municipio = (
    municipio
    .merge(dim_ibge, on="id_municipio", how="left")           # nome, UF e região reais (IBGE)
    .merge(
        metas_municipio.groupby("id_municipio", as_index=False)[colunas_meta[1:]].mean(),
        on="id_municipio",
        how="left",
    )
)
assert len(indicador_municipio) == len(municipio), "join municipal explodiu linhas"

escreve_silver(municipio, "municipio")
escreve_silver(metas_municipio, "metas_municipio")
escreve_silver(indicador_municipio, "indicadores_municipio_integrado")
indicador_municipio.head(3)

Ausentes removidos (sem taxa): 0
[qualidade] silver.municipio: 10/10 checks aprovados
silver/municipio                             23,995 linhas -> data_lake\silver\municipio.parquet
silver/metas_municipio                       10,704 linhas -> data_lake\silver\metas_municipio.parquet
silver/indicadores_municipio_integrado       23,995 linhas -> data_lake\silver\indicadores_municipio_integrado.parquet


,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,...,regiao,percentual_participacao,nivel_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030
0,2023,1100031,2,municipal,69.10,767.8763,NaN,NaN,NaN,NaN,...,Norte,91.535,3.5,70.85,72.53,74.15,75.71,77.21,78.64,80.0
1,2023,1100072,2,municipal,58.20,747.8918,NaN,NaN,NaN,NaN,...,Norte,90.305,3.0,61.82,65.31,68.64,71.79,74.74,77.48,80.0
2,2023,1100189,2,publica,69.73,762.4062,NaN,NaN,NaN,NaN,...,Norte,90.040,4.0,71.37,72.95,74.48,75.95,77.36,78.71,80.0


### 4.3 Metas Brasil

In [16]:
metas_brasil = padroniza_codigos(le_bronze("meta_alfabetizacao_brasil"), ["rede"])

relatorios_qualidade.append(
    DataQualityCheck("silver.metas_brasil", metas_brasil)
    .not_null(["rede", "taxa_alfabetizacao", "meta_alfabetizacao_2030"])
    .unique(["ano", "rede"])
    .intervalo("taxa_alfabetizacao", 0, 100)
    .valida()
)
escreve_silver(metas_brasil, "metas_brasil")
metas_brasil

[qualidade] silver.metas_brasil: 5/5 checks aprovados
silver/metas_brasil                               3 linhas -> data_lake\silver\metas_brasil.parquet


,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao
0,2025,publica,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80.0,88.00
1,2024,publica,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0,87.37
2,2023,publica,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0,86.00


### 4.4 Alunos — unificação batch + streaming

In [17]:
COLUNAS_ALUNOS = [
    "ano", "id_aluno", "id_escola", "id_municipio", "rede", "serie", "caderno",
    "presenca", "preenchimento_caderno", "peso_aluno", "proficiencia", "alfabetizado",
]

alunos_batch = le_bronze("alunos")[COLUNAS_ALUNOS].assign(origem="batch")
alunos_stream = (
    le_bronze("eventos_alunos")[COLUNAS_ALUNOS].assign(origem="streaming")
)

alunos = pd.concat([alunos_batch, alunos_stream], ignore_index=True)
alunos = padroniza_codigos(alunos, ["rede", "serie", "presenca", "alfabetizado"])

# Dados reais do INEP usam códigos (dicionário: presenca 1=presente; alfabetizado 1=sim;
# rede 2=estadual, 3=municipal); simulador/streaming usam rótulos.
# `replace` só toca os códigos — rótulos textuais passam intactos (modo duplo preservado).
alunos["presenca"] = alunos["presenca"].replace({"0": "ausente", "1": "presente"})
alunos["alfabetizado"] = alunos["alfabetizado"].replace({"0": "nao", "1": "sim"})
alunos["rede"] = alunos["rede"].replace(
    {"1": "federal", "2": "estadual", "3": "municipal", "4": "privada"}
)
alunos["id_municipio"] = alunos["id_municipio"].astype(str).str.strip()
alunos["ano"] = alunos["ano"].astype("int64")
alunos["proficiencia"] = pd.to_numeric(alunos["proficiencia"], errors="coerce")

antes = len(alunos)
alunos = alunos[alunos["presenca"] == "presente"].dropna(subset=["proficiencia"])
print(f"Removidos (ausentes ou sem proficiência): {antes - len(alunos)}")

# nulos viraram a string "nan"/"none" em padroniza_codigos; descarte justificado
sem_alfabetizado = alunos["alfabetizado"].isin(["nan", "none", ""])
if sem_alfabetizado.any():
    print(f"Aviso: {sem_alfabetizado.sum()} presentes sem 'alfabetizado' — removidos")
    alunos = alunos[~sem_alfabetizado]

alunos = alunos.drop_duplicates(subset=["ano", "id_aluno"], keep="last")
alunos["alfabetizado_flag"] = alunos["alfabetizado"].eq("sim")
alunos = alunos.merge(dim_ibge[["id_municipio", "nome_municipio", "sigla_uf"]], on="id_municipio", how="left")

relatorios_qualidade.append(
    DataQualityCheck("silver.alunos", alunos)
    .not_null(["ano", "id_aluno", "id_municipio", "proficiencia"])
    .unique(["ano", "id_aluno"])
    .intervalo("proficiencia", 300, 1100)
    .valores_permitidos("alfabetizado", {"sim", "nao"})
    .row_count_min(5000)
    .valida()
)
escreve_silver(alunos, "alunos")
alunos.groupby("origem").size().rename("registros").to_frame()

Removidos (ausentes ou sem proficiência): 513338
[qualidade] silver.alunos: 8/8 checks aprovados
silver/alunos                               3,355,156 linhas -> data_lake\silver\alunos.parquet


,registros
origem,
batch,3354661
streaming,495


In [18]:
relatorio = pd.concat(relatorios_qualidade, ignore_index=True)
caminho_relatorio = QUALITY / f"quality_report_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}.json"
relatorio.to_json(caminho_relatorio, orient="records", force_ascii=False, indent=2)

taxa_aprovacao = 100 * relatorio["passou"].mean()
print(f"Relatório de qualidade salvo em {caminho_relatorio}")
print(f"Score global de qualidade: {taxa_aprovacao:.1f}% ({int(relatorio['passou'].sum())}/{len(relatorio)} checks)")
relatorio

Relatório de qualidade salvo em data_lake\quality_reports\quality_report_20260713T222757.json
Score global de qualidade: 100.0% (33/33 checks)


,tabela,check,coluna,passou,critico,detalhe
0,silver.uf,not_null,ano,True,True,0 nulos
1,silver.uf,not_null,sigla_uf,True,True,0 nulos
2,silver.uf,not_null,rede,True,True,0 nulos
3,silver.uf,not_null,serie,True,True,0 nulos
4,silver.uf,not_null,taxa_alfabetizacao,True,True,0 nulos
5,silver.uf,unique,ano+sigla_uf+rede+serie,True,True,0 duplicados
6,silver.uf,valores_permitidos,sigla_uf,True,True,0 inválidos
7,silver.uf,intervalo,taxa_alfabetizacao,True,True,"0 fora de [0, 100]"
8,silver.uf,intervalo,soma_proporcoes,True,False,"0 fora de [0.98, 1.02]"
9,silver.uf,row_count_min,*,True,True,145 linhas (mínimo 50)


## 5. Gold — camada analítica

Tabelas prontas para consumo por dashboards, análises estatísticas e modelos:

| Tabela | Pergunta que responde |
|---|---|
| `gold_alfabetizacao_municipio` | Cada município atingiu a meta do ano? Qual o gap? |
| `gold_alfabetizacao_uf` | Ranking e evolução temporal das UFs |
| `gold_brasil_evolucao` | Taxa nacional vs metas 2024–2030 |
| `gold_desempenho_alunos` | Desempenho por município/rede/série (atualizada pelo streaming) |

In [19]:
def escreve_gold(df: pd.DataFrame, nome: str, particao=None) -> None:
    destino = GOLD / nome
    if particao:
        df.to_parquet(destino, index=False, partition_cols=particao)
    else:
        destino.mkdir(parents=True, exist_ok=True)
        df.to_parquet(destino / f"{nome}.parquet", index=False)
    print(f"gold/{nome:34s} {len(df):>7,} linhas")


indicador_municipio = pd.read_parquet(SILVER / "indicadores_municipio_integrado.parquet")
indicador_uf = pd.read_parquet(SILVER / "indicadores_uf_integrado.parquet")
alunos_silver = pd.read_parquet(SILVER / "alunos.parquet")
metas_brasil = pd.read_parquet(SILVER / "metas_brasil.parquet")

In [20]:
# 5.1 Município: taxa vs meta do ano da avaliação
def meta_do_ano(df: pd.DataFrame) -> pd.Series:
    """Seleciona a meta correspondente ao ano da avaliação (a partir de 2024)."""
    metas = pd.Series(np.nan, index=df.index)
    for ano_meta in range(2024, 2031):
        selecao = df["ano"] == ano_meta
        metas[selecao] = df.loc[selecao, f"meta_alfabetizacao_{ano_meta}"]
    return metas


gold_municipio = indicador_municipio[
    ["ano", "id_municipio", "nome_municipio", "sigla_uf", "regiao",
     "rede", "rede_desc", "serie", "serie_desc",
     "taxa_alfabetizacao", "media_portugues", "nivel_alfabetizacao"]
].copy()
gold_municipio["meta_ano"] = meta_do_ano(indicador_municipio).round(2)
gold_municipio["gap_meta"] = (gold_municipio["taxa_alfabetizacao"] - gold_municipio["meta_ano"]).round(2)
gold_municipio["atingiu_meta"] = gold_municipio["gap_meta"] >= 0

escreve_gold(gold_municipio, "gold_alfabetizacao_municipio", particao=["ano"])
resumo_metas = (
    gold_municipio.dropna(subset=["meta_ano"])
    .groupby(["ano", "rede_desc"])["atingiu_meta"]
    .agg(municipios="size", pct_atingiu="mean")
)
resumo_metas["pct_atingiu"] = (100 * resumo_metas["pct_atingiu"]).round(1)
resumo_metas

gold/gold_alfabetizacao_municipio        23,995 linhas


municipios  pct_atingiu
ano  rede_desc                                                              
2024 Estadual                                               922         53.0
     Municipal                                             5232         53.3
     Pública (Estadual e Municipal)                        5232         53.8
     Total (Federal, Estadual, Municipal e Privada)         385         18.7

In [21]:
# 5.2 UF: ranking anual e evolução temporal
gold_uf = (
    indicador_uf[indicador_uf["rede"] == "publica"]
    .groupby(["ano", "sigla_uf"], as_index=False)
    .agg(taxa_alfabetizacao=("taxa_alfabetizacao", "mean"),
         media_portugues=("media_portugues", "mean"),
         meta_2030=("meta_alfabetizacao_2030", "mean"))
    .round(2)
)
gold_uf["ranking_nacional"] = (
    gold_uf.groupby("ano")["taxa_alfabetizacao"].rank(ascending=False, method="min").astype(int)
)
gold_uf = gold_uf.sort_values(["sigla_uf", "ano"])
gold_uf["variacao_anual_pp"] = (
    gold_uf.groupby("sigla_uf")["taxa_alfabetizacao"].diff().round(2)
)

escreve_gold(gold_uf, "gold_alfabetizacao_uf")
gold_uf.sort_values(["ano", "ranking_nacional"]).groupby("ano").head(5)

gold/gold_alfabetizacao_uf                   49 linhas


,ano,sigla_uf,taxa_alfabetizacao,media_portugues,meta_2030,ranking_nacional,variacao_anual_pp
4,2023,CE,84.46,795.68,80.0,1,NaN
15,2023,PR,73.12,757.21,80.0,2,NaN
5,2023,ES,67.93,753.72,80.0,3,NaN
6,2023,GO,66.74,752.15,80.0,4,NaN
18,2023,RO,64.60,759.44,80.0,5,NaN
29,2024,CE,85.31,797.29,80.0,1,0.85
31,2024,GO,72.74,759.01,80.0,2,6.00
33,2024,MG,72.07,758.22,80.0,3,12.26
30,2024,ES,71.69,758.52,80.0,4,3.76
40,2024,PR,70.42,754.66,80.0,5,-2.70


In [22]:
# 5.3 Brasil: taxa nacional observada vs trajetória de metas 2024–2030
observado = (
    indicador_uf[indicador_uf["rede"] == "publica"]
    .groupby("ano", as_index=False)["taxa_alfabetizacao"]
    .mean()
    .round(2)
    .assign(tipo="observado")
)
metas_longas = (
    metas_brasil.loc[
        metas_brasil["rede"] == "publica",
        [f"meta_alfabetizacao_{a}" for a in range(2024, 2031)],
    ]
    .melt(var_name="ano", value_name="taxa_alfabetizacao")
    .assign(
        ano=lambda d: d["ano"].str.extract(r"(\d{4})").astype(int),
        tipo="meta",
    )
)
gold_brasil = pd.concat([observado, metas_longas], ignore_index=True).sort_values(["tipo", "ano"])

escreve_gold(gold_brasil, "gold_brasil_evolucao")
gold_brasil.pivot_table(index="ano", columns="tipo", values="taxa_alfabetizacao")

gold/gold_brasil_evolucao                    23 linhas


tipo,meta,observado
ano,,
2023,NaN,54.25
2024,59.933333,56.61
2025,63.846667,NaN
2026,67.313333,NaN
2027,70.980000,NaN
2028,74.153333,NaN
2029,77.160000,NaN
2030,80.000000,NaN


In [23]:
# 5.4 Alunos: desempenho por município/rede/série — atualizada pelo streaming
gold_alunos = (
    alunos_silver
    .groupby(["ano", "id_municipio", "nome_municipio", "sigla_uf", "rede", "serie", "origem"], dropna=False)
    .agg(
        n_alunos=("id_aluno", "nunique"),
        pct_alfabetizados=("alfabetizado_flag", "mean"),
        proficiencia_media=("proficiencia", "mean"),
        peso_medio=("peso_aluno", "mean"),
    )
    .reset_index()
)
gold_alunos["pct_alfabetizados"] = (100 * gold_alunos["pct_alfabetizados"]).round(2)
gold_alunos[["proficiencia_media", "peso_medio"]] = gold_alunos[["proficiencia_media", "peso_medio"]].round(2)

escreve_gold(gold_alunos, "gold_desempenho_alunos", particao=["ano"])
gold_alunos.groupby("origem").agg(
    grupos=("n_alunos", "size"),
    alunos=("n_alunos", "sum"),
    pct_alfabetizados=("pct_alfabetizados", "mean"),
).round(2)

gold/gold_desempenho_alunos              12,902 linhas


,grupos,alunos,pct_alfabetizados
origem,,,
batch,12417,3354661,62.09
streaming,485,495,62.16


In [24]:
print("Estrutura final do data lake:\n")
for camada in (BRONZE, SILVER, GOLD, QUALITY):
    itens = sorted(p.name for p in camada.iterdir())
    print(f"📂 {camada.relative_to(BASE_DIR)}: {len(itens)} itens")
    for item in itens:
        print(f"   · {item}")

Estrutura final do data lake:

📂 bronze: 10 itens
   · alunos
   · dicionario
   · dim_municipio_ibge
   · eventos_alunos
   · eventos_indicadores
   · meta_alfabetizacao_brasil
   · meta_alfabetizacao_municipio
   · meta_alfabetizacao_uf
   · municipio
   · uf
📂 silver: 8 itens
   · alunos.parquet
   · indicadores_municipio_integrado.parquet
   · indicadores_uf_integrado.parquet
   · metas_brasil.parquet
   · metas_municipio.parquet
   · metas_uf.parquet
   · municipio.parquet
   · uf.parquet
📂 gold: 4 itens
   · gold_alfabetizacao_municipio
   · gold_alfabetizacao_uf
   · gold_brasil_evolucao
   · gold_desempenho_alunos
📂 quality_reports: 2 itens
   · quality_report_20260713T204059.json
   · quality_report_20260713T222757.json


## 6. FinOps e decisões de arquitetura

**Otimização de custos aplicada neste pipeline:**

- **Parquet colunar + compressão snappy** em todas as camadas: leitura seletiva de colunas e
  ~5–10× menos bytes que CSV/JSON — em nuvem, custo de armazenamento e de varredura de
  queries (Athena/BigQuery cobram por bytes lidos) cai na mesma proporção;
- **Particionamento**: Bronze por data de ingestão (reprocessos e auditoria leem só a partição
  do dia), Gold por `ano` (dashboards filtram por ano sem varrer o histórico);
- **Camadas medalhão como controle de custo**: consumidores analíticos leem a Gold (pequena e
  agregada) em vez de reprocessar a Bronze; a fonte só é consultada na ingestão;
- **Modo simulado com esquema real**: desenvolvimento e testes com custo zero de nuvem;
  a troca para produção é um único flag (`USE_BIGQUERY = True`).

**Trade-offs:**

| Decisão | Alternativa | Por quê |
|---|---|---|
| Batch para históricos + streaming só para eventos novos | Tudo streaming | Metas e históricos mudam raramente; streaming contínuo custa caro e não agrega frescor útil |
| Data lake (Parquet) | Data warehouse direto | Lake preserva o bruto barato e flexível; o warehouse entra como camada de consumo da Gold |
| Micro-lotes simulados | Kafka/Kinesis real | Mesma semântica para validação local; o barramento real entra na fase cloud |

**Evolução para nuvem:** os jobs AWS Glue deste repositório (`etl-bronze.py`, `etl-silver.py`,
`etl-gold.py`) já implementam este mesmo desenho sobre S3 (buckets SoR/SoT/Spec) — a migração
consiste em apontar a extração para a Base dos Dados/BigQuery e portar as transformações
desta versão pandas (ou da versão PySpark, no notebook irmão) para os jobs.